In [1]:
import networkx as nx
import numpy as np
import ot
from gwgraphs import *

In [2]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("BZR_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("BZR_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("BZR_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("BZR_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    return edges, graph_indicators, graph_labels, node_labels


def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels):
    graphs = {}
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)

    for node_id, (graph_id,
                  node_label) in enumerate(zip(graph_indicators, node_labels),
                                           start=1):
        graphs[graph_id].nodes[node_id]['label'] = node_label

    return graphs


def compute_averages(graphs):
    # Initialize variables
    total_nodes = 0
    class_nodes = {}
    class_counts = {}
    # Iterate through graphs to compute totals
    for graph_id, graph in graphs.items():
        num_nodes = graph.number_of_nodes()
        total_nodes += num_nodes
        
        # Get the class label for the current graph
        graph_label = graph.graph['label']
        
        if graph_label not in class_nodes:
            class_nodes[graph_label] = 0
            class_counts[graph_label] = 0
        
        class_nodes[graph_label] += num_nodes
        class_counts[graph_label] += 1
    # Calculate total average
    total_avg = round(total_nodes / len(graphs))
    
    # Calculate class-based averages
    class_averages = {label: round(class_nodes[label] / count)
                      for label, count in class_counts.items()}
    
    return total_avg, class_averages


def compute_bzr_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix

In [3]:
edges, graph_indicators, graph_labels, node_labels = read_graph_data()
graphs = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                      node_labels)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )
total_avg, class_averages = compute_averages(graphs)
print(f"Average number of nodes in entire dataset: {total_avg}")
print("Average number of nodes for each class:")
for class_label, avg_nodes in class_averages.items():
        print(f"  Class {class_label}: {avg_nodes}")

Graph ID: 1, number of nodes: 30, label: -1
Graph ID: 2, number of nodes: 33, label: -1
Graph ID: 3, number of nodes: 30, label: -1
Graph ID: 4, number of nodes: 33, label: -1
Graph ID: 5, number of nodes: 30, label: -1
Graph ID: 6, number of nodes: 33, label: -1
Graph ID: 7, number of nodes: 30, label: -1
Graph ID: 8, number of nodes: 30, label: -1
Graph ID: 9, number of nodes: 33, label: -1
Graph ID: 10, number of nodes: 30, label: -1
Graph ID: 11, number of nodes: 30, label: -1
Graph ID: 12, number of nodes: 30, label: -1
Graph ID: 13, number of nodes: 33, label: -1
Graph ID: 14, number of nodes: 33, label: -1
Graph ID: 15, number of nodes: 33, label: -1
Graph ID: 16, number of nodes: 33, label: -1
Graph ID: 17, number of nodes: 32, label: -1
Graph ID: 18, number of nodes: 32, label: -1
Graph ID: 19, number of nodes: 32, label: -1
Graph ID: 20, number of nodes: 35, label: -1
Graph ID: 21, number of nodes: 32, label: -1
Graph ID: 22, number of nodes: 35, label: -1
Graph ID: 23, numbe

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_bzr_distance_matrix(graph) for graph in graphs_list]
# Determine number of elements of each class
print("Elements of Class 1: ", labels_list.count(1))
print("Elements of Class -1: ", labels_list.count(-1))

Elements of Class 1:  86
Elements of Class -1:  319


In [5]:
### kNN with BZR data ###

In [6]:
k_list = [1,2,3,4,5,6,7,8,9,10]
num_folds = 10

In [7]:
accuracies, by_class_accuracies, f1_scores, best_k_info = gw_knn(distance_matrices, labels_list, num_folds, k_list=k_list, method="stratk")

Stratified k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 33.33333333333333%, Fold 0
k=1, Class -1 Accuracy: 96.875%, Fold 0
k=1, Overall Classification Accuracy: 82.92682926829268%, Fold 0
k=1, F1 Score (weighted avg): 80.26%, Fold 0
k=2, Class 1 Accuracy: 33.33333333333333%, Fold 0
k=2, Class -1 Accuracy: 96.875%, Fold 0
k=2, Overall Classification Accuracy: 82.92682926829268%, Fold 0
k=2, F1 Score (weighted avg): 80.26%, Fold 0
k=3, Class 1 Accuracy: 11.11111111111111%, Fold 0
k=3, Class -1 Accuracy: 96.875%, Fold 0
k=3, Overall Classification Accuracy: 78.04878048780488%, Fold 0
k=3, F1 Score (weighted avg): 72.15%, Fold 0
k=4, Class 1 Accuracy: 11.11111111111111%, Fold 0
k=4, Class -1 Accuracy: 93.75%, Fold 0
k=4, Overall Classification Accuracy: 75.60975609756098%, Fold 0
k=4, F1 Score (weighted avg): 70.56%, Fold 0
k=5, Class 1 Accuracy: 11.11111111111111%, Fold 0
k=5, Class -1 Accuracy: 93.75%, Fold 0
k=5, Overall Classification Accuracy: 75.60975609756098%, Fold 0
k=5, F1 S

In [8]:
### kNN with gLGW distance ###
## CITATION ###
# https://github.com/Gorgotha/LGW #
# lgw_procedure, LGW_graph, and LGW_eucl are taken directly from this code.
# k_folds_glgw_knn is a modified function from the above code. Everything else is my work.

In [9]:
# Normalize the matrices
Ms = []
for matrix in distance_matrices:
    Ms.append(matrix / np.max(matrix))

In [10]:
k_bary = 36  # use overall dataset average
heights = [
    np.full(len(graph), 1 / len(graph)) for graph in graphs_list
]
num_folds = 10

accuracies2, by_class_accuracies2, f1_scores2, best_k_info2 = k_folds_glgw_knn(distance_matrices,
            labels_list,
            k_bary,
            Ms,
            heights,
            num_folds,
            seed=42,
            k_knn_list=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            method="stratk")

Stratified k-fold CV selected
Fold 0:
Time taken for reference barycenter computation: 1.9028 seconds
Time taken for LGW computation: 2.5554 seconds
k=1, Class 1 Accuracy: 33.33333333333333%
k=1, Class -1 Accuracy: 93.75%
k=1, Overall Classification Accuracy: 80.48780487804879%
k=1, F1 Score (weighted avg): 78.27%
k=2, Class 1 Accuracy: 33.33333333333333%
k=2, Class -1 Accuracy: 93.75%
k=2, Overall Classification Accuracy: 80.48780487804879%
k=2, F1 Score (weighted avg): 78.27%
k=3, Class 1 Accuracy: 33.33333333333333%
k=3, Class -1 Accuracy: 84.375%
k=3, Overall Classification Accuracy: 73.17073170731707%
k=3, F1 Score (weighted avg): 72.59%
k=4, Class 1 Accuracy: 33.33333333333333%
k=4, Class -1 Accuracy: 87.5%
k=4, Overall Classification Accuracy: 75.60975609756098%
k=4, F1 Score (weighted avg): 74.45%
k=5, Class 1 Accuracy: 0.0%
k=5, Class -1 Accuracy: 93.75%
k=5, Overall Classification Accuracy: 73.17073170731707%
k=5, F1 Score (weighted avg): 65.96%
k=6, Class 1 Accuracy: 11.1111